# Word2Vec + Logistic Regression

This notebook trains Word2Vec context embeddings and evaluates a per-word Logistic Regression classifier for word sense prediction.

In [ ]:
import re

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score

## Load Data

In [ ]:
df = pd.read_csv("data/train/train.csv", sep="\t")
df = df.drop(columns=["predict_sense_id"], errors="ignore")

print(f"Rows: {len(df)}")
print(f"Words: {df['word'].nunique()}")
print(f"Missing positions: {df['positions'].isna().sum()}")
df.head()

## Run Word2Vec + Logistic Regression

In [ ]:
random_states = [42]
valid_frac = 0.2
token_pattern = re.compile(r"[а-яёa-z0-9]+", re.IGNORECASE)

runs = []
word_scores = []

for random_state in random_states:
    rng = np.random.RandomState(random_state)
    train_idx = []
    valid_idx = []

    for _, word_df in df.groupby("word"):
        for _, sense_df in word_df.groupby("gold_sense_id"):
            indices = sense_df.index.to_numpy().copy()
            rng.shuffle(indices)

            if len(indices) < 2:
                train_idx.extend(indices.tolist())
                continue

            n_valid = max(1, int(round(len(indices) * valid_frac)))
            n_valid = min(n_valid, len(indices) - 1)

            valid_idx.extend(indices[:n_valid].tolist())
            train_idx.extend(indices[n_valid:].tolist())

    train_df = df.loc[sorted(train_idx)].copy()
    valid_df = df.loc[sorted(valid_idx)].copy()

    tokenized_contexts = []
    for context in train_df["context"].fillna(""):
        tokens = token_pattern.findall(str(context).lower())
        tokenized_contexts.append(tokens)

    word2vec = Word2Vec(
        sentences=tokenized_contexts,
        vector_size=100,
        window=5,
        min_count=1,
        workers=1,
        sg=1,
        epochs=30,
        seed=random_state,
    )
    train_vectors = []
    for context in train_df["context"].fillna(""):
        tokens = token_pattern.findall(str(context).lower())
        vectors = []
        for token in tokens:
            if token in word2vec.wv:
                vectors.append(word2vec.wv[token])
        if len(vectors) == 0:
            train_vectors.append(np.zeros(word2vec.vector_size, dtype=np.float32))
        else:
            train_vectors.append(np.mean(vectors, axis=0))
    x_train_w2v = np.vstack(train_vectors)

    valid_vectors = []
    for context in valid_df["context"].fillna(""):
        tokens = token_pattern.findall(str(context).lower())
        vectors = []
        for token in tokens:
            if token in word2vec.wv:
                vectors.append(word2vec.wv[token])
        if len(vectors) == 0:
            valid_vectors.append(np.zeros(word2vec.vector_size, dtype=np.float32))
        else:
            valid_vectors.append(np.mean(vectors, axis=0))
    x_valid_w2v = np.vstack(valid_vectors)

    word2vec_predictions = pd.Series(index=valid_df.index, dtype="object")
    for word in sorted(valid_df["word"].unique()):
        train_mask = (train_df["word"] == word).to_numpy()
        valid_mask = (valid_df["word"] == word).to_numpy()
        y_train = train_df.loc[train_mask, "gold_sense_id"].astype(str)

        if y_train.nunique() == 1:
            pred = np.repeat(y_train.iloc[0], valid_mask.sum())
        else:
            model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state)
            model.fit(x_train_w2v[train_mask], y_train)
            pred = model.predict(x_valid_w2v[valid_mask])

        word2vec_predictions.loc[valid_df.index[valid_mask]] = pred

    scored_df = valid_df[["word", "gold_sense_id"]].copy()
    scored_df["prediction"] = word2vec_predictions.astype(str)
    scored_df["gold_sense_id"] = scored_df["gold_sense_id"].astype(str)

    run_word_scores = []
    for word, word_df in scored_df.groupby("word"):
        ari = adjusted_rand_score(word_df["gold_sense_id"], word_df["prediction"])
        run_word_scores.append((word, ari, len(word_df)))

    scores_df = pd.DataFrame(run_word_scores, columns=["word", "ari", "n_valid"])
    ari = scores_df["ari"].mean()

    runs.append(("word2vec_logreg", random_state, ari))

    scores_df["model"] = "word2vec_logreg"
    scores_df["random_state"] = random_state
    word_scores.append(scores_df)

runs_df = pd.DataFrame(runs, columns=["model", "random_state", "ari"])
word_scores_df = pd.concat(word_scores, ignore_index=True)
runs_df.head()

## Result

In [ ]:
summary_df = (
    runs_df.groupby("model", as_index=False)
    .agg(
        ari=("ari", "mean"),
    )
    .sort_values("ari", ascending=False)
)
summary_df

Word2Vec + Logistic Regression result with `random_state=42`:

```text
word2vec_logreg  ARI 0.0670
```